# CP1 · Examen blanc de la partie I — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/checkpoints/partie_1/03_examen_solutions.ipynb)

L'enregistrement des réponses que vérifie le notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`), calculées à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher. Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

🧪 workbookMaths : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | scipy 1.16.3 | sympy 1.14.0 | matplotlib 3.10.0 | pandas 2.2.3 | scikit-learn 1.6.1
   Graine : 42 | FAST_MODE : True


## Les réponses vérifiées après l'examen

In [2]:
# The answers are computed from the numbers of the statements (01_examen_sujet.md)
import math
from fractions import Fraction


def H(*probs):
    """Entropy in bits of a law given by its probabilities (0 log 0 counts as 0)."""
    return -sum(p * math.log2(p) for p in probs if p > 0)


# CP1.3: the joint law of (X, Y), X = 0, 1, 2 (rows), Y = 0, 1 (columns)
JOINT_3 = {(0, 0): 0.1, (0, 1): 0.2, (1, 0): 0.2, (1, 1): 0.1, (2, 0): 0.1, (2, 1): 0.3}
E_X3 = sum(x * p for (x, y), p in JOINT_3.items())
E_Y3 = sum(y * p for (x, y), p in JOINT_3.items())
V_X3 = sum(x * x * p for (x, y), p in JOINT_3.items()) - E_X3 ** 2
V_Y3 = sum(y * y * p for (x, y), p in JOINT_3.items()) - E_Y3 ** 2
COV_3 = sum(x * y * p for (x, y), p in JOINT_3.items()) - E_X3 * E_Y3

# CP1.4: 50 lengths, their sum and the sum of the squared deviations
N_4, SUM_4, SS_4 = 50, 1230, 1960
MEAN_4 = SUM_4 / N_4
S_4 = math.sqrt(SS_4 / (N_4 - 1))                 # ddof = 1
SE_4 = S_4 / math.sqrt(N_4)
P_4 = 18 / 50
SE_P4 = math.sqrt(P_4 * (1 - P_4) / N_4)

# CP1.7: Newton on x³ + x − 3 from 1, and a log-sum-exp
NEWTON_7 = [1.0]
for _ in range(2):
    x = NEWTON_7[-1]
    NEWTON_7.append(x - (x ** 3 + x - 3) / (3 * x ** 2 + 1))
SCORES_7 = [712, 715, 710]
LSE_7 = max(SCORES_7) + math.log(sum(math.exp(s - max(SCORES_7)) for s in SCORES_7))

# CP1.9: the frequencies p, the old model q, and the joint law of (F, Y)
P_9, Q_9 = (0.45, 0.4, 0.15), (0.6, 0.35, 0.05)
CROSS_9 = -sum(p * math.log2(q) for p, q in zip(P_9, Q_9))
H_Y_F9 = 0.4 * H(0.35 / 0.4, 0.05 / 0.4) + 0.6 * H(0.10 / 0.6, 0.50 / 0.6)

# CP1.12: Newton on g(lam) = lam − 1.6 (1 − e^(−lam)) from 1.6, then the root
XBAR_12 = 32 / 20
NEWTON_12 = [1.6]
for _ in range(30):
    lam = NEWTON_12[-1]
    NEWTON_12.append(lam - (lam - XBAR_12 * (1 - math.exp(-lam))) / (1 - XBAR_12 * math.exp(-lam)))
LAM_12 = NEWTON_12[-1]

**CP1.1 — Intégrales et densité : une aire, une fonction de répartition, un quantile**

In [3]:
wbm.record("CP1.1a", 4.0, decimals=3, mistakes={"par parties : ∫u v' = [u v] − ∫u' v, avec le signe moins": 8 * math.e - 4, "une primitive de e^(x/2) est 2 e^(x/2) : dérive-la pour vérifier": math.e + 1})
wbm.record("CP1.1b", Fraction(1, 4), kind="fraction", simplified=True, mistakes={"intègre les deux termes de x + 1 sur [0 ; 2]": "1/2", "c est l'inverse de l'aire sous x + 1 : l'aire totale de f doit valoir 1": "4"})
wbm.record("CP1.1c", "(x**2 + 2*x)/8", kind="expr", variables=("x",), mistakes={"multiplie par la constante c : F(2) doit valoir 1": "x**2/2 + x", "F est la primitive de f qui vaut 0 en 0, pas f elle-même": "(x + 1)/4", "F(0) doit valoir 0 : choisis la primitive qui s'annule en 0": "(x + 1)**2/8"})
wbm.record("CP1.1d", Fraction(1) - Fraction(3, 8), kind="fraction", simplified=True, mistakes={"P(1 < X ≤ 2) = F(2) − F(1)": "3/8", "c'est une différence de valeurs de la fonction de répartition, pas de la densité": "1/4"})
wbm.record("CP1.1e", -1 + math.sqrt(5), decimals=3, mistakes={"la médiane vérifie F(m) = 1/2 ; ce n'est pas l'espérance": 7 / 6, "la médiane est dans [0 ; 2] : garde la racine de ce côté": -1 - math.sqrt(5), "la densité n'est pas constante : la médiane n'est pas le milieu de l'intervalle": 1, "résous F(m) = 1/2 avec la fonction de répartition de c), constante comprise": -1 + math.sqrt(2)})
wbm.record("CP1.1f", Fraction(1, 4) * (Fraction(8, 3) + Fraction(2)), kind="fraction", simplified=True, mistakes={"E[X] = ∫ x f(x) dx : la densité n'est pas constante": "1", "multiplie par la constante c": "14/3"})

WBM_ANSWER {"coarse_any": {"0": ["d2215451685d172d8d03a3b7b627100c4bf5f619146b439206f7d8d08ecc7c0f"]}, "coarser": {"1": ["2e259616f1f83adf502f5e8f61dd7d3944306cced6c4249b6a98660b5d3569f2"]}, "decimals": 3, "hash": "f6ac76b40d1346ea75b6e5755f7829188546f46337d1358cfa8206012abb4c0f", "hash_coarse": "120440461dbfa8672753c18181bc32185c421f20748a54a2801b7ea5f9c5c874", "id": "CP1.1a", "kind": "float", "magnitude": 0, "mistakes": {"64d1dff2cf44db59be9045d1774fd76af217b4facf052aa7a7c2e2b889741910": "une primitive de e^(x/2) est 2 e^(x/2) : dérive-la pour vérifier", "aa7f0d27f74db923e1029cb476cc51e0496a2010a919ea8296c5fced3fec5852": "par parties : ∫u v' = [u v] − ∫u' v, avec le signe moins"}, "sign": 1}
📝 Ex CP1.1a : réponse enregistrée (float, 3 décimales).


WBM_ANSWER {"hash": "467ef6d09b51c9c4222f620607a86ea3e633b5cb4ee9d6183e1c58f2089013a7", "id": "CP1.1b", "kind": "fraction", "mistakes": {"93c1269ac44f6d0ebbd6824b90b41f35c17730b451e66f1d1dbb14a1c7f20e66": "c est l'inverse de l'aire sous x + 1 : l'aire totale de f doit valoir 1", "b3514980388518da480f5ff44b9f3a758b0cc9117e81ae1d463d8d1a0af55133": "intègre les deux termes de x + 1 sur [0 ; 2]"}, "sign": 1, "simplified": true}
📝 Ex CP1.1b : réponse enregistrée (fraction).


WBM_ANSWER {"hash": "0c4de926d2307e28dc4313997c2dfe2047a8730d81b314b0b8259429f9dd0722", "id": "CP1.1c", "kind": "expr", "mistakes": {"1412e1985c4269fb456afa99e1ebdf5d1ecf2de9437f0a70d7600dabe7cb159f": "F est la primitive de f qui vaut 0 en 0, pas f elle-même", "2af3fb0a663ceabd0dbfe6aaa7931db90ed8c2c787c6abba163375ba68756e59": "F(0) doit valoir 0 : choisis la primitive qui s'annule en 0", "60341bedb35e154def5e5d411e96645a26b3e734639b9bab607aae40287cd242": "multiplie par la constante c : F(2) doit valoir 1"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex CP1.1c : réponse enregistrée (expr).
WBM_ANSWER {"hash": "2b25e3a99591576d8cab3cc94f0878cbeb96922cd812578440ca6643674daa3b", "id": "CP1.1d", "kind": "fraction", "mistakes": {"477e008132fbf7d37f67d6bee56b8a183c382964f343c88bae739af5dd8f4aec": "P(1 < X ≤ 2) = F(2) − F(1)", "7ebf17a879ff87d5e9fb2c6fa7d5ad2069f9d6508bc4b199b5a2f60eaa86ff8a": "c'est une différence de valeurs de la fonction de répartition,

**CP1.2 — Lois usuelles : binomiale, Poisson, normale**

In [4]:
wbm.record("CP1.2a", 0.9 ** 12, decimals=4, mistakes={"aucune erreur sur une case a la probabilité 1 − 0,1, pour chacune des 12 cases": 0.1 ** 12})
wbm.record("CP1.2b", 1 - 0.9 ** 12 - 12 * 0.1 * 0.9 ** 11, decimals=4, mistakes={"« au moins deux » exclut aussi le cas d'une seule erreur": 1 - 0.9 ** 12, "l'erreur unique peut être sur n'importe laquelle des 12 cases : le coefficient binomial": 1 - 0.9 ** 12 - 0.1 * 0.9 ** 11, "« au moins deux » compte aussi 3, 4… erreurs : passe par l'événement contraire": 66 * 0.01 * 0.9 ** 10})
wbm.record("CP1.2c", math.exp(-1.8) * 1.8 ** 2 / 2, decimals=4, mistakes={"la loi de Poisson divise par k!": math.exp(-1.8) * 1.8 ** 2})
wbm.record("CP1.2d", 1 - math.exp(-1.8 / 3), decimals=4, mistakes={"en 20 minutes, le nombre moyen d'alertes n'est plus 1,8 : proportionnel à la durée": 1 - math.exp(-1.8), "20 minutes sont un tiers d'heure : le paramètre est proportionnel à la durée": 1 - math.exp(-1.8 / 20)})
wbm.record("CP1.2e", 1 - 0.9772, decimals=3, mistakes={"1,5 est l'écart-type : z = (x − μ)/σ, sans élever au carré": 0.0912, "P(X < 11) ne regarde qu'un côté de la cloche": 2 * (1 - 0.9772)})
wbm.record("CP1.2f", 14 - 1.2816 * 1.5, decimals=2, mistakes={"dépassée par 90 % des batteries : seules 10 % font moins, la valeur est sous la moyenne": 14 + 1.2816 * 1.5, "90 % au-dessus laisse 10 % en dessous : cherche z tel que Φ(z) = 0,10": 14 - 1.645 * 1.5})

WBM_ANSWER {"coarse_any": {"1": ["42a5c3578ecdc9f8951849162f2d1f712f708eaf3932070ad32f615a6e900826"]}, "coarser": {"2": ["4582a2ad6450aa0251474aea8cfff785d993870847f077967620b50997007e14"]}, "decimals": 4, "hash": "c5a63883c808045a7eeff71e0264b6ca97e90aa0a121d8b7504f6202f26f4c9b", "hash_coarse": "df6d90aa8c19e3a9a70a06713c6609e5e41515515633d41997f656c4b6f50af2", "id": "CP1.2a", "kind": "float", "magnitude": -1, "mistakes": {"bd69c8239c8b3d9db751f19333ca4be39ba241ede694761c4c72a1a60310218a": "aucune erreur sur une case a la probabilité 1 − 0,1, pour chacune des 12 cases"}, "sign": 1}
📝 Ex CP1.2a : réponse enregistrée (float, 4 décimales).
WBM_ANSWER {"coarse_any": {"1": ["71c8999161ac9b673c055ef24bd19b9f7c01bda38739493cf524273a3f824e93"]}, "coarser": {"2": ["9ecaede61ae156d543e4b9a217ce6bb34387ef269786175cd3d1232bc2e51065"]}, "decimals": 4, "hash": "431640fed126c37f3b5fe8d0eca1a7d0d8316788bdc1bab460e5a955843602ac", "hash_coarse": "29498d51804aa073a94090843539404f3bb8ed1b4b314747389700f0

**CP1.3 — Un couple de variables : marginales, indépendance, variance d'une somme**

In [5]:
wbm.record("CP1.3a", [0.3, 0.3, 0.4], decimals=2, mistakes={"la loi de X additionne les deux cases de chaque ligne": [0.1, 0.2, 0.1]})
wbm.record("CP1.3b", False, mistakes={"compare chaque case au produit de ses deux marginales": True})
wbm.record("CP1.3c", COV_3, decimals=2, mistakes={"la covariance est E[XY] − E[X] E[Y]": COV_3 + E_X3 * E_Y3, "X et Y ne sont pas indépendantes : calcule E[XY] avec la loi jointe": 0})
wbm.record("CP1.3d", V_X3, decimals=2, mistakes={"V(X) = E[X²] − E[X]² : retranche le carré de l'espérance": V_X3 + E_X3 ** 2, "c'est l'espérance ; la variance mesure l'écart à l'espérance": E_X3})
wbm.record("CP1.3e", V_X3 + V_Y3 + 2 * COV_3, decimals=2, mistakes={"X et Y ne sont pas indépendantes : la variance d'une somme ajoute 2 Cov(X, Y)": V_X3 + V_Y3, "la variance d'une somme compte la covariance deux fois": V_X3 + V_Y3 + COV_3})

WBM_ANSWER {"decimals": 2, "element_coarse": [["0bf2e44fc179b08acca528e9bf9d78d4838bf9ac83875a7c1439bb2eb855ce97"], ["3b55e1d0350299d33faf19b753caed6246787f9bf50bcc0b3ace75d42a835af1"], ["2027b95cb06a7441b92b8cd2dfcd5750c68aedb5e401e5ca05fb93ea6e0b252a"]], "element_coarse_any": [[], [], []], "element_hashes": ["00376ae70da3247bd5bb4201d79df32b07f5c15aad64d7ca37acd3d86da0e845", "91fdb82bf311d10e80065f1a40cd5a9c5912eef42e58f165aa9835f96c002612", "6412e14f04dda1da6b8c9799f1572a6320c0d814b87a91b49ab4e18fb66638c2"], "hash": "e838409fb674efb5a868d4641a2e47a861ef1e21da30efc696771d060e214a5d", "id": "CP1.3a", "kind": "array", "mistakes": {"6d1a52694e190323f6bf595a716cfd833d0e06adabd51f335e5b0a43d56e1f50": "la loi de X additionne les deux cases de chaque ligne"}, "shape": [3]}
📝 Ex CP1.3a : réponse enregistrée (array, 2 décimales).
WBM_ANSWER {"hash": "67ef418b325576e4a100619559b7a11fe89d2194d6397583ee75ab21b8032ba4", "id": "CP1.3b", "kind": "bool", "mistakes": {"e2c964dae47d04771be7404084a320d

**CP1.4 — Estimer une moyenne : erreur type et intervalle de confiance**

In [6]:
wbm.record("CP1.4a", MEAN_4, decimals=1, mistakes={"la moyenne divise la somme par n, pas par n − 1": SUM_4 / (N_4 - 1)})
wbm.record("CP1.4b", S_4, decimals=3, mistakes={"ddof = 1 : divise la somme des carrés des écarts par n − 1": math.sqrt(SS_4 / N_4), "c'est la variance ; l'écart-type est sa racine carrée": SS_4 / (N_4 - 1)})
wbm.record("CP1.4c", SE_4, decimals=3, mistakes={"l'erreur type divise l'écart-type par √n, pas par n": S_4 / N_4, "l'erreur type part de l'écart-type s, pas de la variance": SS_4 / (N_4 - 1) / N_4, "prends l'écart-type avec ddof = 1": math.sqrt(SS_4 / N_4) / math.sqrt(N_4)})
wbm.record("CP1.4d", [MEAN_4 - 1.96 * SE_4, MEAN_4 + 1.96 * SE_4], decimals=2, mistakes={"l'intervalle de la moyenne utilise l'erreur type, pas l'écart-type des données": [MEAN_4 - 1.96 * S_4, MEAN_4 + 1.96 * S_4], "à 95 %, le quantile est 1,96": [MEAN_4 - 1.645 * SE_4, MEAN_4 + 1.645 * SE_4]})
wbm.record("CP1.4e", 4 * N_4, mistakes={"la largeur varie comme 1/√n : pour la diviser par 2, que faut-il faire à n ?": 2 * N_4, "un intervalle plus étroit demande plus de données, pas moins": N_4 // 4})
wbm.record("CP1.4f", [P_4 - 1.96 * SE_P4, P_4 + 1.96 * SE_P4], decimals=3, mistakes={"l'erreur type d'une proportion est √(p̂ (1 − p̂) / n)": [P_4 - 1.96 * P_4 * (1 - P_4) / N_4, P_4 + 1.96 * P_4 * (1 - P_4) / N_4]})

WBM_ANSWER {"decimals": 1, "hash": "12411b07b4b06f8e8c54b4fc76f31becdf8fecf9dbfa30474367e02d2fdeb191", "hash_coarse": "c016c1fa6aabaed70726787155a6a011df92e1d8fffb76edd48277caf6080af1", "id": "CP1.4a", "kind": "float", "magnitude": 1, "mistakes": {"ddbb1d45989cb96e67c58bdc5622e0c3687038e90ee167fe068f7d405740e4fa": "la moyenne divise la somme par n, pas par n − 1"}, "sign": 1}
📝 Ex CP1.4a : réponse enregistrée (float, 1 décimale).
WBM_ANSWER {"coarse_any": {"0": ["5eabfb02ff9294d32bb56e69764efdf7740ad0b5191d0c1129bce16127eaf864"]}, "coarser": {"1": ["0eec1c5bada728a668f45313a8ca2d076d35af1cad55448d7d4e5e5b7eef7d98"]}, "decimals": 3, "hash": "6f1dda8634791d7b496b534aab90fc239b87cd4834ec0d6e7ad7c78522ce4693", "hash_coarse": "2be5d1bd34cbff06c000849852a306f9168a0787321c4243e29843244c774bfa", "id": "CP1.4b", "kind": "float", "magnitude": 0, "mistakes": {"3013d850c28d87a82f35add9ab89bac9b7db210a6af7fafbc6bdc5926f197e6a": "c'est la variance ; l'écart-type est sa racine carrée", "850c0771581a2

**CP1.5 — Un tableau de contingence et des métriques comme probabilités**

In [7]:
wbm.record("CP1.5a", Fraction(45 + 30, 500), kind="fraction", simplified=True, mistakes={"les alertes sont tous les événements signalés, vrais et faux": "9/100", "c'est la part des vraies chutes ; la question demande la part des alertes": "3/25"})
wbm.record("CP1.5b", Fraction(45, 45 + 30), kind="fraction", simplified=True, mistakes={"la precision conditionne par l'alerte : parmi les alertes, la part de vraies chutes": "3/4"})
wbm.record("CP1.5c", Fraction(45, 60), kind="fraction", simplified=True, mistakes={"le recall conditionne par la vérité : parmi les vraies chutes, la part qui a donné l'alerte": "3/5"})
wbm.record("CP1.5d", 2 * Fraction(3, 5) * Fraction(3, 4) / (Fraction(3, 5) + Fraction(3, 4)), kind="fraction", simplified=True, mistakes={"le F1 est la moyenne harmonique de la precision et du recall, pas leur moyenne arithmétique": "27/40", "F1 = 2PR / (P + R) : n'oublie pas le dénominateur": "9/20"})
wbm.record("CP1.5e", 410 / 440, decimals=4, mistakes={"la spécificité conditionne par l'absence de chute : les 440 autres événements": 410 / 455})
wbm.record("CP1.5f", Fraction(60, 500), kind="fraction", simplified=True, mistakes={"une alerte indépendante de ce qui s'est passé ne sait rien : parmi ses alertes, quelle part de chutes ?": "3/5", "c'est la proportion d'alertes ; la question demande la precision": "3/20"})

WBM_ANSWER {"hash": "26caff0426f213fda19d7a28511fe1bf43d7286f87d2ff70c991034298713a85", "id": "CP1.5a", "kind": "fraction", "mistakes": {"27a07922c9b794bb01d30e3efef3deb87b8e7a296a6b2a03af180503acaa8847": "c'est la part des vraies chutes ; la question demande la part des alertes", "5f324467f9073034066bf6b09a6afe2c59068a886d1472f0c9877a956e8b35fa": "les alertes sont tous les événements signalés, vrais et faux"}, "sign": 1, "simplified": true}
📝 Ex CP1.5a : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "2931bfffb41f1a12526c6d237039145d850383cb0ddb862ba75b0ab478be6fc4", "id": "CP1.5b", "kind": "fraction", "mistakes": {"247ca9a6e8c9d8ed69ceb625a1c3c56133b7ff990b880be795d0419c3ae819cf": "la precision conditionne par l'alerte : parmi les alertes, la part de vraies chutes"}, "sign": 1, "simplified": true}
📝 Ex CP1.5b : réponse enregistrée (fraction).
WBM_ANSWER {"hash": "ea49282eb23a8215b2ef858c0178c6af858133b37bd0bca02d7509d776607310", "id": "CP1.5c", "kind": "fraction", "mistakes": {"

**CP1.6 — Bayes et la loi Beta : une mise à jour conjuguée**

In [8]:
wbm.record("CP1.6a", Fraction(3, 10), kind="fraction", simplified=True, mistakes={"la moyenne de Beta(a, b) est a / (a + b)": "3/7", "c'est le mode ; la moyenne est a / (a + b)": "1/4"})
wbm.record("CP1.6b", [3 + 7, 7 + 23], mistakes={"b reçoit le nombre d'échecs, pas le nombre d'essais": [10, 37], "a reçoit les succès, b les échecs": [30, 10], "ajoute les succès et les échecs aux paramètres de l'a priori": [7, 23]})
wbm.record("CP1.6c", Fraction(10, 40), kind="fraction", simplified=True, mistakes={"la moyenne a posteriori tient compte de l'a priori : a / (a + b) avec les paramètres de b)": "7/30", "c'est le mode ; la moyenne est a / (a + b)": "9/38"})
wbm.record("CP1.6d", Fraction(10 - 1, 40 - 2), kind="fraction", simplified=True, mistakes={"le mode de Beta(a, b) est (a − 1) / (a + b − 2)": "1/4", "le MAP tient compte de l'a priori : c'est le mode de la loi a posteriori": "7/30"})
wbm.record("CP1.6e", Fraction(13, 50), kind="fraction", simplified=True, mistakes={"l'a priori de la deuxième mise à jour est l'a posteriori de la première": "3/10", "ajoute aussi les 10 nouvelles propositions": "1/4"})
wbm.record("CP1.6f", 13 / 50 * 14 / 51, decimals=4, mistakes={"la deuxième acceptation se calcule avec la loi mise à jour par la première, pas avec la même moyenne": (13 / 50) ** 2, "la question demande que les deux soient acceptées": 13 / 50})

WBM_ANSWER {"hash": "4a9c4c3496a026c02ef9f1fedc9d062206f319d3417c7fb15062b7d0807fe320", "id": "CP1.6a", "kind": "fraction", "mistakes": {"310680ed2d8ee82fea73173eb7a14890b734cb18b20917ae303d4154b1758aa3": "la moyenne de Beta(a, b) est a / (a + b)", "4f71a5cb0c7ff7ac143cd0c5a349301b57312c96f1daffa12a8a5dc2d03939f7": "c'est le mode ; la moyenne est a / (a + b)"}, "sign": 1, "simplified": true}
📝 Ex CP1.6a : réponse enregistrée (fraction).
WBM_ANSWER {"decimals": 0, "element_hashes": ["fd6e2cd8d6d83a18390cb59c71fba51b27663b4b93333f17879c93d2d741bf32", "875343ce93b0e2b78e90b1ca71db44fec892e0312e063c554ad4ff9411c4a301"], "hash": "cba747fd02b1a47d394a651eccb4cb4a03aa6c229e299ac278beb115f61de57d", "id": "CP1.6b", "integer": true, "kind": "array", "mistakes": {"53ef97295d8f19da7e1ffbd9b76ea1dae4d91c515dd280b39e4ee5125990fe5c": "b reçoit le nombre d'échecs, pas le nombre d'essais", "b8b46a5bfeeb5cc9538c9e595104f5898b04d1583fe877d0f80fc5aa5a8d937c": "ajoute les succès et les échecs aux paramètre

**CP1.7 — Taylor, Newton et un calcul stable**

In [9]:
wbm.record("CP1.7a", "2 + x/4 - x**2/64", kind="expr", variables=("x",), mistakes={"le terme d'ordre 2 est f''(0) x² / 2 : n'oublie pas le 2": "2 + x/4 - x**2/32", "dérive √(4 + x) avec la règle de la chaîne, puis prends x = 0": "2 + x/2 - x**2/8", "l'ordre 2 ajoute un terme en x²": "2 + x/4"})
wbm.record("CP1.7b", 2 + 0.4 / 4 - 0.4 ** 2 / 64, decimals=4, mistakes={"l'ordre 2 ajoute un terme en x²": 2.1, "le terme d'ordre 2 est f''(0) x² / 2 : n'oublie pas le 2": 2 + 0.4 / 4 - 0.4 ** 2 / 32, "c'est la valeur exacte ; la question demande celle que donne le polynôme": math.sqrt(4.4)})
wbm.record("CP1.7c", NEWTON_7[1], decimals=4, mistakes={"le pas de Newton est f(x)/f'(x), pas l'inverse": 1 - 4 / (-1), "x₁ = x₀ − f(x₀)/f'(x₀) : attention au signe de f(1)": 0.75})
wbm.record("CP1.7d", NEWTON_7[2], decimals=4, mistakes={"c'est (presque) la racine ; la question demande le deuxième pas, seulement": 1.213411662762183})
wbm.record("CP1.7e", LSE_7, decimals=3, mistakes={"le log-sum-exp vaut le maximum plus le logarithme d'une somme d'exponentielles des écarts au maximum": 715})
wbm.record("CP1.7f", math.exp(715 - LSE_7), decimals=4, mistakes={"la softmax prend l'exponentielle des scores avant de normaliser": 715 / 2137, "c'est la probabilité d'un autre score : relis lequel est 715": math.exp(712 - LSE_7)})

WBM_ANSWER {"hash": "08750caf423d6003298837db6ea55fbf1346203324aea67b47a85f27dcef85cb", "id": "CP1.7a", "kind": "expr", "mistakes": {"4405c5e8a68358f3525a01e0d9c2f8ad9419bf31ac86d22da8b3389b56e85315": "dérive √(4 + x) avec la règle de la chaîne, puis prends x = 0", "e541dd20eaf8b24dce8c9316c050d3f178c51f41a2cb96f8ab198ec5cf307e30": "le terme d'ordre 2 est f''(0) x² / 2 : n'oublie pas le 2", "f91aabffcfc3e2f84a478e71fb054e9ac5173af67bfaef0757389035c5b9ab38": "l'ordre 2 ajoute un terme en x²"}, "points": [["1/3"], ["5/7"], ["6/5"], ["-2/5"], ["9/4"]], "variables": ["x"]}
📝 Ex CP1.7a : réponse enregistrée (expr).
WBM_ANSWER {"coarse_any": {"0": ["f4591760c99ea21ff7fa75a19202e7be2a16bcaca4a6f74e2834cd986ef1ee78"]}, "coarser": {"1": ["90b839c156b47216eea749a18aa036e434e1d3f4e8ceb4ce96fbe0efae2e965d"], "2": ["4fb6bdd066fd73513e51bd01a347607dc7cd2b7fbd0606b4edd1a5411704400e"]}, "decimals": 4, "hash": "0627d8b2adad55a6b3e06076c23c2bbd218422db432e822ce9556ecd1587a96f", "hash_coarse": "09eaef6a7

**CP1.8 — Gradient, dérivée directionnelle et point critique**

In [10]:
wbm.record("CP1.8a", [3 * 1 ** 2 - 6 * 2, -6 * 1 + 6 * 2], mistakes={"∂f/∂x dérive aussi −6xy, où y est une constante": [3, 6], "le gradient est [∂f/∂x, ∂f/∂y], dans cet ordre": [6, -9]})
wbm.record("CP1.8b", Fraction(-9 * 8 + 6 * 15, 17), kind="fraction", simplified=True, mistakes={"divise le vecteur direction par sa norme : la direction est un vecteur unitaire": "18"})
wbm.record("CP1.8c", math.hypot(-9, 6), decimals=3, mistakes={"la plus grande pente est la norme du gradient, une racine carrée": 117, "la norme est √(a² + b²), pas |a| + |b|": 15})
wbm.record("CP1.8d", [0, 0, 2, 2], mistakes={"range les points dans l'ordre des x croissants": [2, 2, 0, 0]})
wbm.record("CP1.8e", 3, choices=[1, 2, 3], mistakes={"en (0, 0), calcule f_xx f_yy − (f_xy)² : son signe décide": 1})
wbm.record("CP1.8f", 2 ** 3 - 6 * 2 * 2 + 3 * 2 ** 2, mistakes={"c'est la valeur en (0, 0) ; la question porte sur l'autre point critique": 0, "attention au signe du terme −6xy": 44})

WBM_ANSWER {"decimals": 0, "element_hashes": ["e19a9aabd8d6a73d347198d916a2a098aa572cafb5e10da50c29b33f6bfb172e", "53b9f617a9f01a404c7383fc0724f39508aa97613c22775ec2a8b0eb34206797"], "hash": "07bbe95f28e797ba397db75a3f0584e5a63e430424d92d4b60aba61333fa53c3", "id": "CP1.8a", "integer": true, "kind": "array", "mistakes": {"3c9404f649ec5e806327006d970f4f86378edd96551f3c91469da4f9ffbba762": "∂f/∂x dérive aussi −6xy, où y est une constante", "9c341813dd08d86464f9b8531353fb07e2ab83f00d47214d2e01d6d79b39d180": "le gradient est [∂f/∂x, ∂f/∂y], dans cet ordre"}, "shape": [2]}
📝 Ex CP1.8a : réponse enregistrée (array, 0 décimale).
WBM_ANSWER {"hash": "214ee95e328a61e0a3bf439d574a90cc64f575ff37eb4a09fe29652e8cfb6f15", "id": "CP1.8b", "kind": "fraction", "mistakes": {"4e697a098841f91fa739dcd650a03ca09ebaec65ce516b5be2f04af9b208b2cf": "divise le vecteur direction par sa norme : la direction est un vecteur unitaire"}, "sign": 1, "simplified": true}
📝 Ex CP1.8b : réponse enregistrée (fraction).
WBM_A

**CP1.9 — Entropie, information mutuelle et KL**

In [11]:
wbm.record("CP1.9a", H(*P_9), decimals=3, mistakes={"les entropies sont en bits : log₂, pas ln": H(*P_9) * math.log(2)})
wbm.record("CP1.9b", CROSS_9, decimals=3, mistakes={"H(p, q) pondère par p les longueurs −log₂ q": -sum(q * math.log2(p) for p, q in zip(P_9, Q_9)), "c'est l'entropie de q ; la question demande la cross-entropy": H(*Q_9)})
wbm.record("CP1.9c", CROSS_9 - H(*P_9), decimals=3, mistakes={"KL(p ‖ q) pondère par p : l'ordre des deux lois compte": sum(q * math.log2(q / p) for p, q in zip(P_9, Q_9)), "une divergence KL est positive ou nulle": H(*P_9) - CROSS_9})
wbm.record("CP1.9d", Fraction(35, 40), kind="fraction", simplified=True, mistakes={"P(question | F = 1) divise par P(F = 1), la somme de la ligne": "7/9", "c'est la probabilité jointe ; une probabilité conditionnelle divise par P(F = 1)": "7/20"})
wbm.record("CP1.9e", H_Y_F9, decimals=3, mistakes={"H(Y | F) est une moyenne des entropies des lignes, pondérée par P(F), pas leur somme": H(0.875, 0.125) + H(1 / 6, 5 / 6), "pondère par P(F = 1) et P(F = 0), pas par 1/2": (H(0.875, 0.125) + H(1 / 6, 5 / 6)) / 2, "c'est H(F) ; la question demande H(Y | F)": H(0.4, 0.6)})
wbm.record("CP1.9f", H(0.45, 0.55) - H_Y_F9, decimals=3, mistakes={"I(F ; Y) = H(Y) − H(Y | F) : c'est la baisse de l'incertitude": H_Y_F9, "H(Y) se calcule avec la loi de Y, la somme de chaque colonne": 1 - H_Y_F9})

WBM_ANSWER {"coarse_any": {"0": ["5de5ea8746d3f50b2933701297f846b50cac1dc2d555986ddd49d1c8e2cce433"]}, "coarser": {"1": ["25d26b77479a29937e0b12c2a9a4cfe77a82a2bff83c4ea9981a06b23b9e1749"]}, "decimals": 3, "hash": "161a1c52c5b6a3ebd653da61d2547bfdcf0536faa00b4375c0a76cb06ece3537", "hash_coarse": "f570f925fef87ee644619a11d67385668936b5b0d3893fc6a940e2e6618093e7", "id": "CP1.9a", "kind": "float", "magnitude": 0, "mistakes": {"363bd75e59a558a51909a11ae826d26dafc51338360cd10a4419e5707b14986e": "les entropies sont en bits : log₂, pas ln"}, "sign": 1}
📝 Ex CP1.9a : réponse enregistrée (float, 3 décimales).
WBM_ANSWER {"coarse_any": {"0": ["19719b3c3c30aae928a90350a751abfc987b66e210cde204868ba7ca80f9f57f"]}, "coarser": {"1": ["d95baa53495585b913663347397760db32928a28bcc796d39183882362023cef"]}, "decimals": 3, "hash": "726c6ad1837efcb7baaa174619ed22ac2c6bbc389e6535311327430d98ca4fa5", "hash_coarse": "a7e86cefed33856bf5563b16807a66ffb7befac10cf270eb8e5b2eec08b065a6", "id": "CP1.9b", "kind": "fl

**CP1.11 — Corriger une copie : cinq erreurs (les lettres seulement)**

In [12]:
wbm.record("CP1.11a", "ACDFG", mistakes={"il manque une des lignes fausses : il y en a exactement cinq": {'ACDF', 'ACDG', 'ACFG', 'ADFG', 'CDFG'}, "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise": {'ABCDFG', 'ACDEFG', 'ACDFGH'}, "une ligne juste est citée à la place d'une ligne fausse": {'ABCDF', 'ABCDG', 'ABCFG', 'ABDFG', 'ACDEF', 'ACDEG', 'ACDFH', 'ACDGH', 'ACEFG', 'ACFGH', 'ADEFG', 'ADFGH', 'BCDFG', 'CDEFG', 'CDFGH'}})

WBM_ANSWER {"hash": "48fe8c8413cd9c33e7033d5e499f714bb8f474dea9c3ddcd780ea3cd85440a86", "id": "CP1.11a", "kind": "str", "mistakes": {"078eb36b4eec518cd3e9c3a465ddad778d2b4689198a452312a13e872fe6b721": "une ligne juste est citée à la place d'une ligne fausse", "10a6452ff428042d2ce1a98b75d112f5bbf7faecf6e2846dfaf90b65f257dff9": "une des lignes citées est juste : relis-les une par une, avec la règle qu'elle utilise", "1224689a0417e3c36ee7112655fdd95ecec9b29ff6c452ecaa0eeea98ced008e": "une ligne juste est citée à la place d'une ligne fausse", "2278dfd86482d449c3721a6ce1fe1b8ff4a7b6a5e0591a2a2d88ab5e35c675b8": "une ligne juste est citée à la place d'une ligne fausse", "40b26f6807b30e54cb65eb9727a8c506e419f2ddf570a737cffa02126503f025": "une ligne juste est citée à la place d'une ligne fausse", "54e4d0652dedc86302e39480b9cae7b19dc56c0031884242f892db66c9a58916": "une ligne juste est citée à la place d'une ligne fausse", "5d14ce61e9dc3667b2797056963a56e2f7c82c3572c8e3684d4c77715a022610": "une l

**CP1.12 — Problème de synthèse : estimer un paramètre de bout en bout**

In [13]:
wbm.record("CP1.12a", math.exp(-1) / (1 - math.exp(-1)), decimals=3, mistakes={"la liste ne garde que les fichiers qui ont au moins un bug : divise par 1 − e^(−λ)": math.exp(-1)})
wbm.record("CP1.12b", "32/lam - 20/(1 - exp(-lam))", kind="expr", variables=("lam",), mistakes={"dérive aussi le terme −20 ln(1 − e^(−λ))": "32/lam - 20", "la dérivée de ln(1 − e^(−λ)) est e^(−λ) / (1 − e^(−λ)) : attention au signe": "32/lam - 20 + 20*exp(-lam)/(1 - exp(-lam))", "la log-vraisemblance somme les 20 fichiers": "1.6/lam - 1/(1 - exp(-lam))"})
wbm.record("CP1.12c", NEWTON_12[1], decimals=4, mistakes={"un pas de Newton retranche g(λ)/g'(λ)": 1.6 + (1.6 - XBAR_12 * (1 - math.exp(-1.6))) / (1 - XBAR_12 * math.exp(-1.6)), "la dérivée de −1,6 (1 − e^(−λ)) est −1,6 e^(−λ)": 1.6 - (1.6 - XBAR_12 * (1 - math.exp(-1.6))) / (1 + XBAR_12 * math.exp(-1.6))})
wbm.record("CP1.12d", NEWTON_12[2], decimals=4, mistakes={"le deuxième pas part de λ₁, avec g et g' recalculés en λ₁": NEWTON_12[1]})
wbm.record("CP1.12e", LAM_12, decimals=3, mistakes={"la moyenne des fichiers de la liste ne compte pas les fichiers sans bug : résous g(λ) = 0": XBAR_12})
wbm.record("CP1.12f", True, mistakes={"calcule ℓ'(1) avec ton expression de b)": False})
wbm.record("CP1.12g", math.exp(-LAM_12), decimals=3, mistakes={"avec l'estimation de e), pas avec la moyenne des fichiers de la liste": math.exp(-XBAR_12)})
wbm.record("CP1.12h", [LAM_12 - 1.96 * 0.28, LAM_12 + 1.96 * 0.28], decimals=2, mistakes={"l'intervalle est centré sur l'estimation de e)": [XBAR_12 - 1.96 * 0.28, XBAR_12 + 1.96 * 0.28], "multiplie l'écart-type du bootstrap par 1,96": [LAM_12 - 0.28, LAM_12 + 0.28]})

WBM_ANSWER {"coarse_any": {"0": ["e83ff021df369169f18be4172b328de3a6cd297f78e6766ad51ee3464a94e438"], "1": ["642e781ce030fbb50ba949d15df167cb980cb3488e42aafb3c85a21b1673e941"]}, "decimals": 3, "hash": "60ade46276d76e1d834efcdec7f494adc8a1d3dd2ad327f7ddc07223ee4f6313", "hash_coarse": "e54aeaba9f579c50040f216b532cb337df799a9906ddb995a48d4d5aa3a5921f", "id": "CP1.12a", "kind": "float", "magnitude": -1, "mistakes": {"1dd49df8f847758d2c2b5648b6be0fbde172b437bb615a59ff1f78951f1b8bf0": "la liste ne garde que les fichiers qui ont au moins un bug : divise par 1 − e^(−λ)"}, "sign": 1}
📝 Ex CP1.12a : réponse enregistrée (float, 3 décimales).
WBM_ANSWER {"hash": "de4091b6811317ef915595d881e3844939336c96141b61bd0d33885c797a254d", "id": "CP1.12b", "kind": "expr", "mistakes": {"2a7614cfc37e1739afd188b0e5eb1a94aacc275268e5d2acfd455abb69d75379": "la log-vraisemblance somme les 20 fichiers", "4ad7ae7e48a8f339698ca09e0cfb2b75f1d4f297f140d6ecb3d225bacf49e23c": "la dérivée de ln(1 − e^(−λ)) est e^(−λ) / (1